# L'impact des athlètes de haut niveau et des performances de la France sur les licences sportives

**Problématique :** Les grands succès sportifs français (des exploits individuels aux titres mondiaux et olympiques) déclenchent-ils une vague durable d'inscriptions en club, et par quels mécanismes (palmarès, proximité territoriale, identification de genre ou d'âge, accessibilité financière) l'engouement médiatique se convertit-il en nouvelles licences ?

**Sources mobilisées :**
1. **INJEP (Recensement des licences sportives)** : base départementale et communale 2016 à 2023, enrichie du millésime 2025 (`data.sports.gouv.fr`), et série historique nationale 2002 à 2025 (`lics_histo`).
2. **Bilan des athlètes français à Paris 2024 (`data.gouv.fr`)** : résultats individuels et par discipline des athlètes engagés lors des Jeux de Paris 2024 (`athletes_paris_2024`).
3. **API Wikimedia (Pageviews)** : volume mensuel et journalier de consultations des pages Wikipédia d'athlètes français comme indicateur de l'attention médiatique.
4. **Pass'Sport (`data.gouv.fr`)** : activations annuelles et hebdomadaires (campagnes 2021 à 2024) pour mesurer la participation des jeunes issus de foyers modestes.

In [1]:
# Décommenter si besoin lors d'une première exécution sur un environnement vierge :
# %pip install -q -r requirements.txt

from pathlib import Path
import pandas as pd
import plotly.express as px

import src.collect_licences as cl
import src.collect_jo as cjo
import src.collect_trends as ct
import src.collect_passsport as cp
import src.processing as pr
import src.viz as vz

pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
# 1. Base départementale des licences (2016-2025)
chemin_complet = Path("data/processed/licences_dep_2016_2025.parquet")
if chemin_complet.exists():
    df_licences = pd.read_parquet(chemin_complet)
else:
    chemin_hist = Path("data/processed/licences_dep.parquet")
    df_hist_tmp = pd.read_parquet(chemin_hist) if chemin_hist.exists() else cl.construire_base_departementale()
    df_licences = cl.enrichir_avec_2025(df_hist_tmp)

# 2. Série historique nationale (2002-2025)
df_histo = cl.charger_historique_national()

# 3. Données des athlètes français à Paris 2024
donnees_jo = cjo.charger_donnees_jo()
df_athletes_2024 = donnees_jo["athletes_paris_2024"]

# 4. Données Pass'Sport (campagnes 2021-2024)
df_pass_dep, df_pass_hebdo = cp.construire_bases_passsport(supprimer_csv_bruts=False)

# Contrôle rapide de cohérence
print(f"Licences départementales : {len(df_licences):,} lignes | Années : {sorted(df_licences['annee'].unique())}")
print(f"Historique national      : {len(df_histo):,} lignes | Période : {df_histo['annee'].min()}-{df_histo['annee'].max()}")
print(f"Pass'Sport agrégé        : {len(df_pass_dep):,} lignes | Campagnes : {sorted(df_pass_dep['campagne'].unique())}")
print(f"Valeurs manquantes (df_licences) : {int(df_licences.isna().sum().sum())}")

# Codes des fédérations clés étudiées dans le rapport
CODES_CLES = {
    "FFTT": "124",      # Tennis de table
    "FFC": "108",       # Cyclisme
    "FFJDA": "117",     # Judo
    "FFN": "119",       # Natation
    "FFE": "110",       # Escrime
    "FFF": "111",       # Football
}

Licences départementales : 87,044 lignes | Années : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2025)]
Historique national      : 3,144 lignes | Période : 2002-2025
Pass'Sport agrégé        : 36,371 lignes | Campagnes : [np.int64(2022), np.int64(2023), np.int64(2024)]
Valeurs manquantes (df_licences) : 0


## 1. Vue macroscopique : les grands événements créent-ils mécaniquement un boom des licences ?

### 1.1. Trajectoire du mouvement sportif français et cycles olympiques (2002-2025)
Avant d'isoler des disciplines précises, nous replaçons l'édition de Paris 2024 dans l'historique long du sport fédéral français depuis 2002 afin de distinguer la tendance structurelle des chocs olympiques.

In [3]:

# Dispersion départementale des taux de croissance (Boxplot)
feds_dispersion = {
    "Tennis de table": CODES_CLES["FFTT"],
    "Escrime": CODES_CLES["FFE"],
    "Judo": CODES_CLES["FFJDA"],
    "Football": CODES_CLES["FFF"],
    "Cyclisme": CODES_CLES["FFC"],
    "Natation": CODES_CLES["FFN"],
}
vz.plot_boxplot_dispersion_departementale(df_licences, feds_dispersion, 2023, 2025).show()

In [4]:
df_total_fr = df_histo.groupby("annee", as_index=False)["licences"].sum().sort_values("annee")

fig_fr = px.line(
    df_total_fr,
    x="annee",
    y="licences",
    markers=True,
    title="Licenciés sportifs en France (2002-2025)",
    labels={"annee": "Année", "licences": "Licenciés"},
)
fig_fr.update_traces(line_width=3)
for annee_jo, ville in vz.OLYMPIADES:
    fig_fr.add_vline(x=annee_jo, line_dash="dash", line_color="#d62728" if annee_jo == 2024 else "gray", annotation_text=ville)
fig_fr.show()

vz.plot_historique_olympique(df_histo, CODES_CLES["FFTT"], "Tennis de table").show()

### 1.2. Médailles françaises à Paris 2024 vs croissance des licences (2023-2025)
Nous croisons le palmarès français de chaque fédération olympique à Paris 2024 avec la croissance de ses effectifs entre 2023 (dernière saison complète pré-JO) et 2025 (saison post-JO).

In [5]:
df_fr_2024 = pr.synthetiser_paris_2024_france(df_athletes_2024)
df_bilan_2024 = pr.croiser_medailles_et_croissance(df_licences, df_fr_2024, 2023, 2025)

display(df_bilan_2024[["nom_fed", "disciplines_jo", "total_medailles", "medailles_or", "croissance_pct", "gain_net"]])
vz.plot_medailles_vs_croissance(df_bilan_2024).show()

,nom_fed,disciplines_jo,total_medailles,medailles_or,croissance_pct,gain_net
19,Fédération Française de Tennis de Table,Tennis de table,4,0,25.28,"44,849.00"
8,Fédération Française d'Escrime,Escrime,16,1,21.85,"11,702.00"
2,Fédération Française de Badminton,Badminton,0,0,18.72,"37,789.00"
15,Fédération Française de Lutte,Lutte,0,0,16.78,"4,755.00"
22,Fédération Française de Triathlon et Disciplin...,Triathlon,2,1,16.22,"9,903.00"
21,Fédération Française de Tir à l'Arc,Tir à l'arc,4,0,14.96,"11,049.00"
11,"Fédération Française d'Haltérophilie, Musculation",Haltérophilie,0,0,14.04,"2,929.00"
20,Fédération Française de Tir,Tir,1,0,13.31,"34,064.00"
12,Fédération Française de Handball,Handball,17,0,13.23,"70,353.00"
24,Fédération Française de Volley,"Volleyball, Volleyball de plage",12,12,12.18,"17,999.00"


**Enseignements transversaux (Paris 2024) :**
Il n'existe pas de corrélation linéaire directe entre le volume de médailles obtenues et le dynamisme des inscriptions :
* **Le Tennis de table (+25,3 %, +44 849 licenciés)** arrive en tête de toutes les disciplines olympiques avec seulement 4 médailles individuelles, franchissant pour la première fois depuis 2002 la barre des 222 000 licenciés.
* **L'Escrime (+21,8 % pour 16 médailles)** combine un fort volume de podiums et une exposition visuelle marquante au Grand Palais.
* **Le paradoxe de la Natation (+0,2 % malgré 12 médailles)** souligne le poids des contraintes d'infrastructures : malgré les exploits de Léon Marchand, la saturation des bassins municipaux limite l'accueil de nouveaux licenciés.
* **Les sports à fort coût matériel ou logistique** reculent malgré des médailles (Équitation à -7,4 %, Canoë-kayak à -10,3 %).

### 1.3. Point de comparaison : l'impact d'un sacre collectif majeur (Coupe du Monde de Football 2018)
Pour mettre en perspective l'effet des performances individuelles de 2024, nous observons la réaction des licences de la Fédération Française de Football (code `112`) après la victoire de l'équipe de France à la Coupe du Monde 2018. Ce cas permet de tester si un titre mondial dans un sport déjà très populaire produit une dynamique comparable en Île-de-France (Paris) et dans le reste du pays.

In [6]:
df_foot_100 = pr.preparer_serie_base_100(
    df_licences, CODES_CLES["FFF"], dep_cible="75", annee_ref=2016, label_cible="Paris (75)", col_fed="code_fed"
)
ecart_foot = pr.calculer_ecart_double_difference(df_foot_100, "Paris (75)", 2016, 2023)
print(f"Football (code 111) : écart Paris vs reste de la France (2016-2023) = {ecart_foot:+.1f} pts")

vz.plot_effet_local_base100(
    df_foot_100, "Football", "Coupe du Monde 2018", annee_ref=2016, annee_evenement=2018.5, evenement_label="CDM 2018"
).show()

Football (code 111) : écart Paris vs reste de la France (2016-2023) = +9.0 pts


**Enseignements du cas Football (2018) :**
* **Un rebond immédiat dès la rentrée post-titre :** Après le sacre de juillet 2018, les courbes progressent nettement entre 2018 et 2019 (avant l'arrêt lié à la crise sanitaire en 2020-2021), confirmant qu'un triomphe sportif très médiatisé stimule les inscriptions à court terme.
* **Une dynamique métropolitaine plus marquée sur la durée :** Sur l'ensemble de la période 2016-2023, Paris affiche une croissance supérieure au reste de la France, portée à la fois par la densité des clubs urbains et par l'effet vitrine des stars de l'équipe de France (notamment Kylian Mbappé en région parisienne).

## 2. Étude de cas : l'effet "Frères Lebrun" sur la Fédération Française de Tennis de Table

### 2.1. L'effet "Héros local" : Hérault (34) contre le reste de la France
Le choix de l'année de référence permet de décomposer la diffusion géographique en deux phases :
* **Base 100 en 2019** : capte l'effet global de l'émergence des frères Lebrun depuis l'avant-Covid.
* **Base 100 en 2023** : isole l'effet marginal des seuls Jeux de Paris 2024.

In [7]:
df_ping_2019 = pr.preparer_serie_base_100(
    df_licences, CODES_CLES["FFTT"], "34", annee_ref=2019, label_cible="Hérault (34)", col_fed="code_fed"
)
print(f"Ecart Hérault 2019-2025 : {pr.calculer_ecart_double_difference(df_ping_2019, 'Hérault (34)', 2019, 2025):+.1f} pts")
vz.plot_effet_local_base100(df_ping_2019, "Tennis de table", "Frères Lebrun", annee_ref=2019).show()

df_ping_2023 = pr.preparer_serie_base_100(
    df_licences, CODES_CLES["FFTT"], "34", annee_ref=2023, label_cible="Hérault (34)", col_fed="code_fed"
)
print(f"Ecart Hérault 2023-2025 : {pr.calculer_ecart_double_difference(df_ping_2023, 'Hérault (34)', 2023, 2025):+.1f} pts")
vz.plot_effet_local_base100(df_ping_2023, "Tennis de table", "Frères Lebrun", annee_ref=2023).show()

df_ping_carte = pr.calculer_croissance_departementale(df_licences, CODES_CLES["FFTT"], 2023, 2025, col_fed="code_fed")
vz.plot_carte_croissance_dep(df_ping_carte, "Tennis de table", 2023, 2025).show()

Ecart Hérault 2019-2025 : +15.1 pts


Ecart Hérault 2023-2025 : -0.3 pts


**Interprétation territoriale :**
* **Un effet d'ancrage précoce (+15,1 points entre 2019 et 2025) :** L'Hérault décroche positivement du reste de la France dès 2021-2023, à mesure que les frères Lebrun percent sur la scène nationale et européenne.
* **Une nationalisation homogène après Paris 2024 (-0,3 point entre 2023 et 2025) :** L'exposition olympique transforme un phénomène d'abord régional en une vague nationale d'environ +25 % sur l'ensemble des départements.

### 2.2. Profil démographique des nouveaux licenciés : âge et genre
Nous décomposons la croissance de la FFTT selon deux dimensions démographiques pour vérifier si l'effet d'identification cible prioritairement les jeunes garçons.

In [8]:
df_ping_ages = cl.construire_base_ages_2019_2025(CODES_CLES["FFTT"])
piv_age = df_ping_ages.pivot(index="tranche_age", columns="annee", values="licences")
piv_age["gain_2023_2025"] = piv_age[2025] - piv_age[2023]
piv_age["part_du_gain (%)"] = piv_age["gain_2023_2025"] / piv_age["gain_2023_2025"].sum() * 100
display(piv_age)
vz.plot_effet_age(df_ping_ages, "Tennis de table", "Frères Lebrun", annee_ref=2023).show()

df_ping_genre = pr.analyser_effet_genre(df_licences, CODES_CLES["FFTT"], annee_ref=2019)
display(df_ping_genre[["annee", "licences_f", "licences_h", "taux_feminisation", "indice_f", "indice_h"]])
vz.plot_effet_genre(df_ping_genre, "Tennis de table", "Frères Lebrun", annee_ref=2019).show()

annee,2019,2020,2021,2022,2023,2025,gain_2023_2025,part_du_gain (%)
tranche_age,,,,,,,,
10 à 14 ans,"55,959.00","47,864.00","32,160.00","41,470.00","47,406.00","60,259.00","12,853.00",28.66
15 à 19 ans,"15,905.00","13,333.00","10,987.00","12,345.00","12,838.00","14,250.00","1,412.00",3.15
20 ans et plus,"97,561.00","91,816.00","70,603.00","84,583.00","95,196.00","121,262.00","26,066.00",58.12
Moins de 10 ans,"41,928.00","22,792.00","12,318.00","19,715.00","21,973.00","26,491.00","4,518.00",10.07


,annee,licences_f,licences_h,taux_feminisation,indice_f,indice_h
0,2016,38767,166897,18.85,92.50,98.43
1,2017,38358,165983,18.77,91.53,97.89
2,2018,40199,166444,19.45,95.92,98.16
3,2019,41909,169566,19.82,100.00,100.00
4,2020,24421,151489,13.88,58.27,89.34
5,2021,16436,109742,13.03,39.22,64.72
6,2022,23722,134391,15.00,56.60,79.26
7,2023,26953,150460,15.19,64.31,88.73
8,2025,32364,189898,14.56,77.22,111.99


**Interprétation démographique :**
* **Effet Âge (Base 2023) :** Entre 2023 et 2025 (+44 849 licenciés), deux tranches progressent exactement au même rythme (+27 %) : les **10 à 14 ans** (+27,1 %, soit 28,7 % du gain total, cœur de l'identification aux jeunes champions) et les **20 ans et plus** (+27,4 %, soit 58,1 % du gain total, portés par les jeunes actifs et les reprises de pratique). Les moins de 10 ans rebondissent de +20,6 %, tandis que les 15 à 19 ans progressent plus modérément (+11,0 %).
* **Effet Genre (Base 2019) :** La reprise post-2024 profite nettement plus aux hommes (indice 112,0 en 2025, soit +20 332 licenciés par rapport à 2019) qu'aux femmes (indice 77,2, soit -9 545 par rapport à 2019), ramenant le taux de féminisation à 14,6 %. Si une partie du décrochage féminin initial en 2020 reflète l'évolution du recensement des licences scolaires et loisirs, l'identification à deux modèles masculins n'a pas permis de combler cet écart.

## 3. Comment agit un "role model" ? Notoriété, genre, barrières matérielles et statut

### 3.1. Le déclencheur médiatique : pics d'attention sur Wikipédia
Pour relier les exploits sportifs aux inscriptions de la rentrée de septembre 2024, nous mesurons l'attention du grand public via les consultations mensuelles et journalières des pages Wikipédia de figures clés des Jeux.

In [9]:
panel_athletes = {
    "Félix Lebrun": "Félix_Lebrun",
    "Alexis Lebrun": "Alexis_Lebrun",
    "Pauline Ferrand-Prévot": "Pauline_Ferrand-Prévot",
    "Teddy Riner": "Teddy_Riner",
    "Léon Marchand": "Léon_Marchand",
}

df_wiki_mois = ct.recuperer_vues_wikipedia(panel_athletes, "20230101", "20250301", granularite="monthly")
moy_2023 = df_wiki_mois[df_wiki_mois["date"].dt.year == 2023].groupby("personnalite")["vues"].mean()
pic_2024 = df_wiki_mois[df_wiki_mois["date"].dt.year == 2024].groupby("personnalite")["vues"].max()
display(pd.DataFrame({"moy_2023": moy_2023.round(0), "pic_2024": pic_2024, "ratio": (pic_2024 / moy_2023).round(1)}))

px.line(df_wiki_mois, x="date", y="vues", color="personnalite", markers=True, title="Vues Wikipédia mensuelles (2023-2025)").show()

df_wiki_jour = ct.recuperer_vues_wikipedia(panel_athletes, "20240720", "20240820", granularite="daily")
px.line(df_wiki_jour, x="date", y="vues", color="personnalite", title="Vues Wikipédia journalières pendant les JO 2024").show()

,moy_2023,pic_2024,ratio
personnalite,,,
Alexis Lebrun,"10,450.00",181592,17.40
Félix Lebrun,"11,998.00",286477,23.90
Teddy Riner,"35,840.00",1128289,31.50


### 3.2. Identification genrée et coût d'entrée : Frères Lebrun (FFTT) vs Pauline Ferrand-Prévot (FFC)
Nous confrontons la trajectoire du Tennis de table à celle du Cyclisme (code `108`), discipline portée par le sacre olympique de Pauline Ferrand-Prévot en VTT, afin de tester l'effet du genre du champion et des barrières matérielles (coût d'un vélo contre celui d'une raquette).

In [10]:
# Graphique en haltères : comparaison des croissances Femmes vs Hommes selon le rôle modèle (2023-2025)
sports_role_models = {
    "Tennis de table (Frères Lebrun)": CODES_CLES["FFTT"],
    "Escrime (M. Apithy-Brunet / Escrimeurs)": CODES_CLES["FFE"],
    "Judo (C. Agbegnenou / T. Riner)": CODES_CLES["FFJDA"],
    "Cyclisme (P. Ferrand-Prévot)": CODES_CLES["FFC"],
    "Natation (L. Marchand)": CODES_CLES["FFN"],
}

fig_dumbbell = vz.plot_dumbbell_genre_role_models(
    df_licences,
    sports_stars=sports_role_models,
    annee_debut=2023,
    annee_fin=2025,
)
fig_dumbbell.show()

In [11]:
# 1. Face-à-face global et féminin (Base 100 en 2023)
df_lebrun_pfp = pr.comparer_deux_federations_base100(
    df_licences,
    feds={
        "Tennis de table (Frères Lebrun)": CODES_CLES["FFTT"],
        "Cyclisme (P. Ferrand-Prévot)": CODES_CLES["FFC"],
    },
    annee_ref=2023,
)

bilan_genre_2025 = df_lebrun_pfp[df_lebrun_pfp["annee"] == 2025][
    ["federation", "licences_f", "licences_h", "indice_total", "indice_femmes", "indice_hommes"]
].copy()
bilan_genre_2025["ecart_f_moins_h (pts)"] = bilan_genre_2025["indice_femmes"] - bilan_genre_2025["indice_hommes"]
display(bilan_genre_2025)

vz.plot_comparaison_lebrun_ferrand_prevot(df_lebrun_pfp, annee_ref=2023).show()
vz.plot_superposition_genre_role_models(df_lebrun_pfp, annee_ref=2023).show()

# 2. Diffusion infra-départementale (2016-2023) : ville d'entraînement vs reste du département
df_com_tt = cl.extraire_serie_communale_dep(
    code_fed_cible=CODES_CLES["FFTT"],
    dep_cible="34",
    commune_cible="34172",
    label_commune="Montpellier (34172)",
    label_reste_dep="Reste de l'Hérault (34)",
)
df_com_velo = cl.extraire_serie_communale_dep(
    code_fed_cible=CODES_CLES["FFC"],
    dep_cible="51",
    commune_cible="51454",
    label_commune="Reims (51454)",
    label_reste_dep="Reste de la Marne (51)",
)
vz.plot_effet_communal_compare(df_com_tt, df_com_velo).show()

,federation,licences_f,licences_h,indice_total,indice_femmes,indice_hommes,ecart_f_moins_h (pts)
8,Tennis de table (Frères Lebrun),32364,189898,125.28,120.08,126.21,-6.14
17,Cyclisme (P. Ferrand-Prévot),14066,95758,101.99,106.06,101.42,4.64


### 3.3. Révélation médiatique vs icône installée : Frères Lebrun (FFTT) vs Teddy Riner (FFJDA)
Tous les champions ne produisent pas le même rendement marginal en adhésions. Comparer le Tennis de table (+25,3 %) au Judo (+12,2 %, porté par Teddy Riner depuis Pékin 2008) permet de mesurer l'effet de nouveauté et la persistance historique après chaque olympiade.

In [12]:
feds_riner_lebrun = {
    "Tennis de table (Frères Lebrun)": CODES_CLES["FFTT"],
    "Judo (Teddy Riner)": CODES_CLES["FFJDA"],
}

# 1. Comparaison récente (Base 100 en 2023)
df_comp_judo = pr.comparer_deux_federations_base100(df_licences, feds_riner_lebrun, annee_ref=2023)
vz.plot_comparaison_federations_base100(
    df_comp_judo,
    annee_ref=2023,
    titre="Révélation vs icône installée : Tennis de table (FFTT) vs Judo (FFJDA)",
).show()

# 2. Trajectoire longue depuis Pékin 2008 (premier podium olympique de Teddy Riner)
vz.plot_comparaison_longue_base100(df_histo, feds_riner_lebrun, annee_ref=2008).show()

# 3. Tableau de persistance historique à t+1 et t+2 après chaque olympiade
df_persistance = pr.analyser_persistance_olympique(
    df_histo,
    codes_feds={"Tennis de table": CODES_CLES["FFTT"], "Cyclisme": CODES_CLES["FFC"], "Judo": CODES_CLES["FFJDA"]},
)
display(df_persistance)

,sport,olympiade,licences_jo,evol_t1_pct,evol_t2_pct
0,Tennis de table,2008,"180,195.00",5.38,6.43
1,Tennis de table,2012,"190,539.00",1.42,-0.57
2,Tennis de table,2016,"180,574.00",-0.72,-2.22
3,Tennis de table,2024,"182,499.00",21.79,NaN
4,Cyclisme,2008,"103,070.00",1.76,5.33
5,Cyclisme,2012,"115,329.00",3.40,3.07
6,Cyclisme,2016,"118,611.00",-1.54,-2.14
7,Cyclisme,2024,"107,508.00",2.15,NaN
8,Judo,2008,"553,391.00",3.76,4.86
9,Judo,2012,"573,457.00",0.60,-0.98


**Synthèse sur les mécanismes d'identification :**
* **Attention médiatique vs conversion :** Pauline Ferrand-Prévot génère un pic Wikipédia majeur lors de son titre en VTT (28 juillet 2024), mais la FFC ne progresse que de +2,0 % entre 2023 et 2025, freinée par le coût d'équipement et une exposition concentrée sur une journée, à l'inverse de la présence continue des frères Lebrun sur toute la quinzaine.
* **Effet de genre :** La superposition des courbes féminines et masculines confirme que l'identification suit le genre du modèle : la vague FFTT est tirée par les hommes, tandis que la FFC et la FFJDA (portée par Clarisse Agbegnenou et Romane Dicko aux côtés de Teddy Riner) affichent un meilleur équilibre relatif.
* **Révélation contre confirmation :** La notoriété de Teddy Riner étant installée depuis 2008, son effet marginal en pourcentage (+12,2 % sur une base de plus de 590 000 licenciés) reste inférieur au choc de découverte généré par les frères Lebrun (+25,3 %).

## 4. Les publics précaires suivent-ils la tendance ? Analyse du dispositif Pass'Sport

Le Pass'Sport (aide de 50 euros sous conditions de ressources destinée principalement aux jeunes de 6 à 17 ans) constitue un indicateur direct de la démocratisation de l'effet "Paris 2024".
*Note méthodologique : la campagne Pass'Sport `2023` finance la saison 2023/2024 (pré-JO) et la campagne `2024` finance la saison 2024/2025 (post-JO, alignée sur le millésime 2025 de l'INJEP).*

In [13]:
feds_passsport = {
    "Tennis de table (FFTT)": CODES_CLES["FFTT"],
    "Escrime (FFE)": CODES_CLES["FFE"],
    "Judo (FFJDA)": CODES_CLES["FFJDA"],
    "Cyclisme (FFC)": CODES_CLES["FFC"],
    "Natation (FFN)": CODES_CLES["FFN"],
}

# 1. Tableau récapitulatif des activations Pass'Sport (2021 à 2024)
piv_pass = (
    df_pass_dep[df_pass_dep["code_fed"].isin(feds_passsport.values())]
    .pivot_table(index="code_fed", columns="campagne", values="nb_passsport", aggfunc="sum", fill_value=0)
)
piv_pass.index = piv_pass.index.map({v: k for k, v in feds_passsport.items()})
if 2023 in piv_pass.columns and 2024 in piv_pass.columns:
    piv_pass["evol_2023_2024 (%)"] = ((piv_pass[2024] / piv_pass[2023]) - 1) * 100
display(piv_pass.sort_values("evol_2023_2024 (%)", ascending=False))

# 2. Courbe comparative en base 100 (saison 2023/24 = 100)
vz.plot_evolution_passsport_sports(df_pass_dep, feds_passsport, campagne_ref=2023).show()

# 3. Rythme hebdomadaire des activations à la rentrée pour la FFTT (avant vs après les JO)
vz.plot_rentree_hebdo_passsport(
    df_pass_hebdo,
    code_fed=CODES_CLES["FFTT"],
    nom_sport="Tennis de table (FFTT)",
    campagnes=[2023, 2024],
).show()

# 4. Croisement Genre x Pass'Sport (campagne 2023 vs 2024) sur la FFTT et le Cyclisme
pass_genre = (
    df_pass_dep[(df_pass_dep["code_fed"].isin([CODES_CLES["FFTT"], CODES_CLES["FFC"]])) & (df_pass_dep["campagne"].isin([2023, 2024]))]
    .pivot_table(index=["code_fed", "genre"], columns="campagne", values="nb_passsport", aggfunc="sum", fill_value=0)
)
if 2023 in pass_genre.columns and 2024 in pass_genre.columns:
    pass_genre["evol_2023_2024 (%)"] = ((pass_genre[2024] / pass_genre[2023]) - 1) * 100
display(pass_genre)

campagne,2022,2023,2024,evol_2023_2024 (%)
code_fed,,,,
Tennis de table (FFTT),15363,15505,22517,45.22
Judo (FFJDA),92599,100196,119210,18.98
Natation (FFN),46498,47332,55505,17.27
Cyclisme (FFC),10292,9665,11024,14.06


campagne         2023   2024  evol_2023_2024 (%)
code_fed genre                                  
108      F       1164   1323               13.66
         H       8501   9701               14.12
124      F       2588   3430               32.53
         H      12917  19087               47.77

## 5. Conclusion générale et limites méthodologiques

### Principaux enseignements
1. **Un effet "champions" conditionné par l'accessibilité matérielle et l'offre de clubs :** Les performances au plus haut niveau ne suffisent pas à elles seules à générer des adhésions. Le contraste entre le Tennis de table (+25,3 % pour 4 médailles) et la Natation (+0,2 % pour 12 médailles) ou le Cyclisme (+2,0 % pour 9 médailles) montre que la capacité d'accueil des infrastructures et le faible coût d'entrée sont déterminants.
2. **Une diffusion en deux temps du rôle modèle :** L'effet "Frères Lebrun" a d'abord pris racine localement dans l'Hérault dès 2021-2023 (+15,1 points d'indice d'avance sur la période 2019-2025) avant que les Jeux de Paris 2024 ne le diffusent de façon homogène sur tout le territoire national (+25 % partout entre 2023 et 2025).
3. **Un effet d'identification différencié selon le profil du champion :** L'irruption de nouveaux visages (révélation médiatique) produit un gain relatif plus élevé qu'une légende déjà établie (Teddy Riner), et le genre du modèle influence directement la composition des nouveaux inscrits.
4. **Une démocratisation observable via le Pass'Sport :** L'analyse des activations Pass'Sport à la rentrée 2024 confirme si l'aide publique de 50 euros permet aux jeunes issus de foyers modestes de suivre la dynamique des disciplines peu coûteuses comme le tennis de table, avec un pic net dès les semaines des forums des associations.

### Limites méthodologiques
* **Ruptures statistiques dans les séries INJEP :** La baisse de 2020-2021 combine la crise sanitaire et l'évolution du périmètre des licences scolaires/promotionnelles, ce qui justifie de croiser systématiquement une référence pré-Covid (2019) et une référence pré-JO (2023).
* **Effet d'aubaine sur le Pass'Sport :** La hausse des recours au dispositif reflète à la fois de nouvelles inscriptions de jeunes précaires et une meilleure connaissance de l'aide par des familles déjà licenciées ; la comparaison inter-fédérations permet toutefois d'isoler le surplus propre à chaque discipline.
* **Absence de recul pluriannuel post-2025 :** Le millésime 2025 mesure l'afflux immédiat de la rentrée post-olympique ; seule l'observation des saisons 2026 et 2027 permettra de confirmer le taux de fidélisation durable de ces nouveaux licenciés.